In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from pathlib import Path
from torchvision import models, transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# Select Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

Using compute device: cpu


In [2]:
# Update this path to your actual dataset location
DATASET_DIR = Path("/home/gabor-levai/Downloads/AI_VIS/archive (3)")
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
records = []

if not DATASET_DIR.exists():
    print("Warning: Directory not found. Please update DATASET_DIR.")

for filepath in DATASET_DIR.rglob("*"):
    if filepath.is_file() and filepath.suffix.lower() in valid_extensions:
        parts_lower = [p.lower() for p in filepath.parts]
        
        if any("real" in p for p in parts_lower):
            label = "Real"
        elif any("fake" in p for p in parts_lower):
            label = "Fake"
        else:
            label = "Unknown"
            
        records.append({"filepath": str(filepath), "label": label})

df = pd.DataFrame(records)
df_filtered = df[df["label"].isin(["Real", "Fake"])].copy()

# We only need the test set for Black-Box Evaluation
if len(df_filtered) > 0:
    _, test_df = train_test_split(df_filtered, test_size=0.2, random_state=42, stratify=df_filtered["label"])
    print(f"Test dataset ready: {len(test_df)} images.")
else:
    print("No valid images found. Check your dataset path.")
    test_df = pd.DataFrame()

Test dataset ready: 409 images.


In [3]:
# Base transformation for evaluation
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Initialize ResNet-18
print("Loading ResNet-18 model...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# ResNet-18 uses 'fc' (fully connected) for its final classification layer, not 'classifier'
model.fc = nn.Linear(model.fc.in_features, 1)

# If you have already trained and saved a ResNet-18 model, uncomment the next two lines to load its weights!
# if os.path.exists('best_resnet18.pth'):
#     model.load_state_dict(torch.load('best_resnet18.pth', map_location=device))

model = model.to(device)
model.eval() # Set to evaluation mode
print("ResNet-18 successfully loaded and pushed to device.")

Loading ResNet-18 model...
ResNet-18 successfully loaded and pushed to device.


In [4]:
def apply_blackbox_corruption(image_path, corruption_type, strength):
    img = Image.open(image_path).convert('RGB')
    
    if strength == 0:
        return base_transform(img)
        
    if corruption_type == 'jpeg':
        quality = max(1, 100 - (strength * 20))
        img.save("temp.jpg", "JPEG", quality=quality)
        img = Image.open("temp.jpg")
    elif corruption_type == 'blur':
        img = img.filter(ImageFilter.GaussianBlur(radius=strength))
    elif corruption_type == 'noise':
        img_arr = np.array(img)
        noise = np.random.normal(0, strength * 15, img_arr.shape)
        img_arr = np.clip(img_arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(img_arr)
    elif corruption_type == 'resize':
        orig_size = img.size
        scale = max(0.1, 1.0 - (strength * 0.2)) 
        small_img = img.resize((int(orig_size[0]*scale), int(orig_size[1]*scale)), Image.BILINEAR)
        img = small_img.resize(orig_size, Image.BILINEAR)
        
    return base_transform(img)

def evaluate_blackbox_robustness(model, df, corruption_type, strength, num_samples=200):
    if len(df) == 0: return 0, 0
    df_sample = df.sample(n=min(num_samples, len(df)), random_state=42).reset_index(drop=True)
    
    correct = 0
    total = len(df_sample)
    confidences = []
    
    with torch.no_grad():
        for idx, row in df_sample.iterrows():
            tensor_img = apply_blackbox_corruption(row['filepath'], corruption_type, strength)
            tensor_img = tensor_img.unsqueeze(0).to(device)
            
            output = model(tensor_img)
            prob = torch.sigmoid(output).item()
            pred = 1.0 if prob >= 0.5 else 0.0
            true_label = 1.0 if row['label'] == 'Real' else 0.0
            
            if pred == true_label:
                correct += 1
            
            conf = prob if true_label == 1.0 else (1.0 - prob)
            confidences.append(conf)
            
    return correct / total, np.mean(confidences)

In [ ]:
corruptions = ['jpeg', 'blur', 'noise', 'resize']
strengths = [0, 1, 2, 3, 4]

results = []

print("Running Black-Box Robustness Sweep on ResNet-18...")
for corruption in corruptions:
    print(f"Testing vulnerability to: {corruption.upper()}")
    for strength in strengths:
        acc, conf = evaluate_blackbox_robustness(
            model=model, 
            df=test_df, 
            corruption_type=corruption, 
            strength=strength, 
            num_samples=200 # Lower this if it takes too long
        )
        
        results.append({
            'Corruption': corruption,
            'Strength': strength,
            'Accuracy': acc,
            'Avg_Confidence': conf
        })
        print(f"  Level {strength} -> Acc: {acc:.1%}, Conf: {conf:.3f}")

results_df = pd.DataFrame(results)
print("\nSweep Complete!")

Running Black-Box Robustness Sweep on ResNet-18...
Testing vulnerability to: JPEG
  Level 0 -> Acc: 50.0%, Conf: 0.498
  Level 1 -> Acc: 49.5%, Conf: 0.499
  Level 2 -> Acc: 50.0%, Conf: 0.498
  Level 3 -> Acc: 47.5%, Conf: 0.497
  Level 4 -> Acc: 48.5%, Conf: 0.499
Testing vulnerability to: BLUR
  Level 0 -> Acc: 50.0%, Conf: 0.498
  Level 1 -> Acc: 49.0%, Conf: 0.497
  Level 2 -> Acc: 47.0%, Conf: 0.499
  Level 3 -> Acc: 47.0%, Conf: 0.499
  Level 4 -> Acc: 47.5%, Conf: 0.499
Testing vulnerability to: NOISE
  Level 0 -> Acc: 50.0%, Conf: 0.498
  Level 1 -> Acc: 48.5%, Conf: 0.496
  Level 2 -> Acc: 46.5%, Conf: 0.496
  Level 3 -> Acc: 43.5%, Conf: 0.493
  Level 4 -> Acc: 48.5%, Conf: 0.497
Testing vulnerability to: RESIZE
  Level 0 -> Acc: 50.0%, Conf: 0.498
  Level 1 -> Acc: 51.0%, Conf: 0.498
  Level 2 -> Acc: 49.5%, Conf: 0.498


In [ ]:
if not results_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Accuracy Degradation
    sns.lineplot(data=results_df, x='Strength', y='Accuracy', hue='Corruption', marker='o', ax=ax1)
    ax1.set_title("ResNet-18 Accuracy Decay by Corruption", fontsize=14)
    ax1.set_ylabel("Accuracy")
    ax1.set_xlabel("Corruption Strength (0 = Clean)")
    ax1.set_xticks(strengths)
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot 2: Confidence Degradation
    sns.lineplot(data=results_df, x='Strength', y='Avg_Confidence', hue='Corruption', marker='s', ax=ax2)
    ax2.set_title("ResNet-18 True-Class Confidence Decay", fontsize=14)
    ax2.set_ylabel("Average Confidence in True Label")
    ax2.set_xlabel("Corruption Strength (0 = Clean)")
    ax2.set_xticks(strengths)
    ax2.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()